# NYC Taxi Trip Duration Prediction

This notebook focuses on building a predictive model to estimate the duration of taxi trips in New York City using the NYC-trip duration dataset.

## 1. Import Libraries

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import urllib.request
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

## 2. Load the Dataset

We will automatically download the dataset if it is not present in the local `Dataset` directory.

In [13]:
DATASET_DIR = 'Dataset'
DATASET_PATH = os.path.join(DATASET_DIR, 'nyc_taxi_trip_duration.csv')
DATASET_RAW_URL = "https://raw.githubusercontent.com/SANJAI-s0/NYC_Taxi_Prediction/main/Dataset/nyc_taxi_trip_duration.csv"

if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

if not os.path.exists(DATASET_PATH):
    print(f"Downloading dataset from Github Raw URL...")
    try:
        urllib.request.urlretrieve(DATASET_RAW_URL, DATASET_PATH)
        print("Download complete!")
    except Exception as e:
        print(f"Error downloading dataset: {e}")
        print("Please ensure the dataset is manually placed in the 'Dataset/' folder or the URL is accessible.")
else:
    print("Dataset already exists locally.")

df = pd.read_csv(DATASET_PATH)
df.head()

Download complete!


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id1080784,2,2016-02-29 16:40:21,2016-02-29 16:47:01,1,-73.953918,40.778873,-73.963875,40.771164,N,400
1,id0889885,1,2016-03-11 23:35:37,2016-03-11 23:53:57,2,-73.988312,40.731743,-73.994751,40.694931,N,1100
2,id0857912,2,2016-02-21 17:59:33,2016-02-21 18:26:48,2,-73.997314,40.721458,-73.948029,40.774918,N,1635
3,id3744273,2,2016-01-05 09:44:31,2016-01-05 10:03:32,6,-73.961670,40.759720,-73.956779,40.780628,N,1141
4,id0232939,1,2016-02-17 06:42:23,2016-02-17 06:56:31,1,-74.017120,40.708469,-73.988182,40.740631,N,848


## 3. Data Preprocessing & Feature Engineering

We will convert datetimes and extract useful time-based features.

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

# Extract features
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['pickup_month'] = df['pickup_datetime'].dt.month

# Calculate Haversine distance
def haversine_distance(lat1, lon1, lat2, lon2):
    r = 6371 # Earth radius in km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * r * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['distance'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'], 
    df['dropoff_latitude'], df['dropoff_longitude']
)

df.head()

## 4. Exploratory Data Analysis (EDA)

### 4.1 Trip Duration Distribution
Notice the outliers. We will use log transformation.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(np.log1p(df['trip_duration']), bins=100, kde=True)
plt.title('Log Transformed Trip Duration Distribution')
plt.xlabel('Log(Trip Duration)')
plt.show()

### 4.2 Distance vs Duration

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='distance', y='trip_duration', data=df[df['trip_duration'] < 5000])
plt.title('Distance vs Trip Duration (Samples < 5000s)')
plt.show()

## 5. Outlier Removal

Remove trips with duration > 24 hours or extremely high speeds (> 100 km/h average).

In [ ]:
# Filter duration between 1 minute and 5 hours
df = df[(df['trip_duration'] > 60) & (df['trip_duration'] < 18000)] 

# Filter distance > 0
df = df[df['distance'] > 0]

print(f"Data size after filtering: {df.shape}")

## 6. Model Building

Let's prepare the features and target.

In [ ]:
features = ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 
            'dropoff_longitude', 'dropoff_latitude', 'pickup_hour', 'pickup_day_of_week', 
            'pickup_month', 'distance']

X = df[features]
y = np.log1p(df['trip_duration']) # Target log-transformed

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

### 6.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print(f"Linear Regression RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr))}")
print(f"Linear Regression MAE: {mean_absolute_error(y_test, y_pred_lr)}")

### 6.2 Decision Tree Regressor

Trees usually handle spatial data and non-linear relationships better.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print(f"Decision Tree RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_dt))}")
print(f"Decision Tree MAE: {mean_absolute_error(y_test, y_pred_dt)}")

## 7. Conclusions

In this project, we explored the NYC Taxi Trip Duration dataset, engineered features like Haversine distance and time components, and built predictive models. 

1. **Distance** is the strongest predictor of trip duration.
2. **Time of day** significantly impacts trip speed due to traffic patterns.
3. **Non-linear models** like Decision Trees perform better than Linear Regression for this dataset.